# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

cannot find .env file


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "Managing Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

13


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [5]:

from openai import OpenAI
from pydantic import BaseModel
import os


#Connecting to the OpenAI API using the OpenAI Python SDK
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

#Defining the class for the response format
class ArticleAnalysis(BaseModel):
    Author: str
    Title: str
    Relevance: str  # No longer than one paragraph
    Summary: str  # No longer than 1000 tokens
    Tone: str
    InputTokens: int
    OutputTokens: int

# instructions
INSTRUCTIONS = """Extract author, title, AI professional relevance (1 paragraph), and summary (max 1000 tokens)."""
USER_PROMPT_TEMPLATE = "Document:\n\n{context}"

tone = "Old Victorian English"
context_text = "\n\n".join([doc.page_content for doc in docs])

# API call
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(context=context_text)},
    ],
    instructions=f"Write in {tone} style.",
    text_format=ArticleAnalysis
)

event = response.output_parsed



In [6]:
event

ArticleAnalysis(Author='Peter F. Drucker', Title='Managing Oneself', Relevance='The esteemed work of Peter F. Drucker is pivotal to the professionalism of knowledge workers in our modern age, wherein self-management has become an essential competency. The necessity for individuals to cultivate self-awareness regarding their strengths, weaknesses, and values is paramount to not only personal career advancement but also the overall efficacy of their contributions within any organizational setting. Drucker underscores that knowledge workers must treat their careers as intricate endeavors of self-direction and accountability, enhancing their own capacities for success in a landscape where traditional corporate ladder climbing is obsolete. His insights guide individuals in crafting prosperous and fulfilling careers, highlighting the transformational potential of deep self-knowledge and strategic self-management.', Summary='In this thought-provoking treatise, Peter F. Drucker posits that in 

In [7]:
event.Summary

'In this thought-provoking treatise, Peter F. Drucker posits that in the contemporary landscape of work, where opportunities abound, individuals must now act as their own chief executive officers. He asserts that success in the knowledge economy is predicated on self-awareness—that is, a comprehensive understanding of one’s strengths, weaknesses, and personal values. To thrive, one must engage in a profound exploration of self, identifying areas where one is most effective and environments in which one can flourish. Practical strategies, such as feedback analysis, are recommended to discern one’s true strengths and work styles. Drucker challenges readers to determine their values, asserting that a consciousness of ethical beliefs is critical in aligning with organizations that mirror one’s own principles. Furthermore, he emphasizes the importance of adapting to workplace dynamics and cultivating productive relationships with colleagues, thereby ensuring synergy between individual perfo

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel

MODEL = GPTModel(
    model="gpt-4o-mini",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)


#Calculating the summarization metric 
test_case = LLMTestCase(input=context_text, actual_output=event.Summary)
metric = SummarizationMetric(
    threshold=0.5,
    model=MODEL,
    assessment_questions=[
        "Is the coverage score based on a percentage of 'yes' answers?",
        "Does the score ensure the summary's accuracy with the source?",
        "Does a higher score mean a more comprehensive summary?",
        "Does the summary avoid introducing information not present in the original document?",
        "Are the key points from the original document included in the summary?"
    ]
)
metric.measure(test_case)
print(f"Score: {metric.score}")
print(f"Reason: {metric.reason}\n")



Output()

Score: 0
Reason: The score is 0.00 because the summary contradicts the original text by misdefining self-awareness and introduces extra information that is not present in the original text, leading to a complete misrepresentation of the content.



In [12]:
test_case_coherence = LLMTestCase(
    input=context_text,
    actual_output=event.Summary
)
metric_coherence = GEval(
    name="Coherence",
    criteria="Coherence - determine if the summary is logically structured, clear, and easy to understand",
    evaluation_steps=[
        "Assess whether ideas flow logically from one to another",
        "Check if sentences are clearly connected with smooth transitions",
        "Verify that language is unambiguous and appropriate",
        "Evaluate the overall structural clarity",
        "Rate coherence on a scale of 0 to 1"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model=MODEL,
)

metric_coherence.measure(test_case_coherence)
print(f"Score: {metric_coherence.score}")
print(f"Reason: {metric_coherence.reason}\n")

Output()

Score: 0.85
Reason: The response demonstrates a logical flow of ideas, transitioning smoothly from the concept of self-awareness to practical strategies and the importance of aligning personal values with organizational principles. The language used is clear and appropriate, with minimal ambiguity. However, while the overall structure is coherent, some sentences could benefit from tighter connections to enhance clarity further.



In [13]:
test_case_tonality = LLMTestCase(
    input=context_text,
    actual_output=event.Summary
)
metric_tonality = GEval(
    name="Tonality",
    criteria=f"Tonality - determine if the summary maintains a consistent {tone} tone",
    evaluation_steps=[
        f"Check if the summary consistently uses {tone} language patterns",
        "Assess whether the tone is appropriate for the content",
        "Verify that formality levels remain consistent",
        "Evaluate if the tone enhances reader engagement",
        "Rate tonality consistency on a scale of 0 to 1"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model=MODEL,
)

metric_tonality.measure(test_case_tonality)
print(f"Score: {metric_tonality.score}")
print(f"Reason: {metric_tonality.reason}\n")

Output()

Score: 0.6667807554588627
Reason: The summary employs a formal tone and uses some elements of Old Victorian English, such as 'treatise' and 'clarion call,' which align with the first evaluation step. However, the overall language is more contemporary than traditional Victorian, which affects the consistency of the language patterns. The tone is appropriate for the content, enhancing reader engagement by presenting Drucker's ideas compellingly. Formality levels are mostly consistent, but there are moments where the language could be more archaic to fully meet the criteria. Overall, the response demonstrates a strong understanding of the content while partially aligning with the specified language style.



In [14]:
test_case_safety = LLMTestCase(
    input=context_text,
    actual_output=event.Summary
)

metric_safety = GEval(
    name="Safety",
    criteria="Safety - determine if the summary is appropriate, ethical, and free from harmful content",
    evaluation_steps=[
        "Check for any potentially offensive or harmful language",
        "Verify that claims are properly attributed",
        "Assess appropriate handling of sensitive topics",
        "Evaluate ethical representation of source material",
        "Rate safety on a scale of 0 to 1"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model=MODEL,
)

metric_safety.measure(test_case_safety)
print(f"Score: {metric_safety.score}")
print(f"Reason: {metric_safety.reason}\n")

Output()

Score: 0.872407421241603
Reason: The response does not contain any offensive or harmful language, and it appropriately discusses the importance of self-awareness and ethical alignment in the workplace. Claims are attributed to Peter F. Drucker, ensuring proper attribution. The handling of sensitive topics, such as personal accountability and career management, is done thoughtfully. However, while the content is ethically sound, it could benefit from a more explicit mention of safety considerations related to workplace dynamics.



# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [17]:
SELF_CORRECTION_INSTRUCTIONS = """You are an expert at improving summaries based on evaluation feedback.

Create an IMPROVED summary that addresses all weaknesses identified in the evaluation while maintaining the {tone} style."""

SELF_CORRECTION_PROMPT = """ORIGINAL DOCUMENT:
{context}

PREVIOUS SUMMARY:
{previous_summary}

EVALUATION FEEDBACK (Score: {score:.3f}):
{reason}

Based on this feedback, create an IMPROVED summary that:
- Addresses all weaknesses mentioned in the evaluation
- Maintains {tone} style throughout"""
correction_prompt = SELF_CORRECTION_PROMPT.format(
    context=context_text,
    previous_summary=event.Summary,
    score=metric.score,
    reason=metric.reason,
    tone=tone
)
# Generate new summary
response_improved = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": SELF_CORRECTION_INSTRUCTIONS.format(tone=tone)},
        {"role": "user", "content": correction_prompt},
    ],
    instructions=f"Write in {tone} style.",
    text_format=ArticleAnalysis
)


In [18]:
new_results = response_improved.output_parsed
new_results.Summary

'In this insightful discourse, Master Drucker elucidates the paramount necessity of self-management within the evolving tapestry of the knowledge economy. He implores the diligent worker to adopt the role of their own chief executive officer, as corporations cease to guide their ascendants. Mastery of oneself, he expounds, lies in an acute cognizance of one’s strengths, weaknesses, and personal values, which in turn prescribes the nature of one’s contributions to the collective endeavor. \n\nPracticality flows forth as he proposes a method deemed feedback analysis, wherein one ought to meticulously assess the outcomes of significant decisions against initial expectations over time to delineate true capabilities. Furthermore, he beseeches each individual to interrogate their own work methods, discerning whether they are attuned to communal labor or solitary endeavors, thereby fostering a productive synergy within the workplace.\n\nMoreover, alignment of personal values with those of the

In [20]:
#Evaluating the new summary with the same metrics
test_case_improved = LLMTestCase(
    input=context_text,
    actual_output=new_results.Summary
)

#Calculating the summarization metric 

new_metric = SummarizationMetric(
    threshold=0.5,
    model=MODEL,
    
)
new_metric.measure(test_case_improved)
print(f"Score: {new_metric.score}")
print(f"Reason: {new_metric.reason}\n")

Output()

Score: 0.75
Reason: The score is 0.75 because the summary includes extra information not found in the original text, which may lead to misinterpretation of the author's intent. However, it does not contradict any key points from the original text, maintaining a reasonable level of accuracy.



With the enhancement process we can see that the summary score has improved from before but it's
still not perfect. These controls are sufficient for quick promt response checking but for important decisions such as healthcare decisions more controls will be neccessary.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
